In [1]:
import sqlite3
import pandas as pd

matches = pd.read_csv("../data/processed/matches_clean.csv")
deliveries = pd.read_csv("../data/processed/deliveries_clean.csv")

conn = sqlite3.connect("../data/processed/ipl.db")
matches.to_sql("matches", conn, if_exists="replace", index=False)
deliveries.to_sql("deliveries", conn, if_exists="replace", index=False)
print("Database created")

Database created


In [2]:
pd.read_sql("""
SELECT winner, COUNT(*) AS wins
FROM matches
GROUP BY winner
ORDER BY wins DESC
""", conn)

,winner,wins
0,Chennai Super Kings,102
1,Mumbai Indians,90
2,Kolkata Knight Riders,83
3,Royal Challengers Bangalore,79
4,Sunrisers Hyderabad,67
5,Rajasthan Royals,65
6,Delhi Capitals,34
7,Kings XI Punjab,31
8,Punjab Kings,24
9,Delhi Daredevils,19


In [3]:
pd.read_sql("""
SELECT toss_decision,
       COUNT(*) AS matches,
       ROUND(100.0 * SUM(toss_winner = winner) / COUNT(*), 1) AS toss_winner_win_pct
FROM matches
GROUP BY toss_decision
""", conn)

,toss_decision,matches,toss_winner_win_pct
0,bat,289,57.4
1,field,305,50.2


In [4]:
pd.read_sql("""
SELECT batter, SUM(batsman_runs) AS runs
FROM deliveries
GROUP BY batter
ORDER BY runs DESC
LIMIT 10
""", conn)

,batter,runs
0,CSK Player 2,6610
1,CSK Player 3,5783
2,RCB Player 2,5614
3,KKR Player 2,5598
4,MI Player 2,5463
5,MI Player 3,5451
6,RCB Player 3,5284
7,CSK Player 4,5191
8,KKR Player 3,5170
9,SRH Player 2,4886


In [5]:
pd.read_sql("""
SELECT m.venue,
       COUNT(*) AS matches,
       ROUND(100.0 * SUM(m.winner = d.batting_team) / COUNT(*), 1) AS chase_win_pct
FROM matches m
JOIN (SELECT DISTINCT match_id, batting_team
      FROM deliveries WHERE inning = 2) d
  ON m.id = d.match_id
GROUP BY m.venue
ORDER BY chase_win_pct DESC
""", conn)

,venue,matches,chase_win_pct
0,Arun Jaitley Stadium,63,55.6
1,Rajiv Gandhi Stadium,72,50.0
2,M Chinnaswamy Stadium,84,48.8
3,IS Bindra Stadium,83,48.2
4,Wankhede Stadium,72,47.2
5,Sawai Mansingh Stadium,68,45.6
6,Eden Gardens,67,41.8
7,MA Chidambaram Stadium,85,36.5
